# Per-channel "ink + one aux" training (5 seeds x 11 channels)

For each seed and each aux channel, train a **fresh** model that physically uses
only the ink channel + that one aux channel (`aux_feat=[channel]`). This isolates
each channel's contribution with a model actually optimized for it, rather than
masking channels at inference time.

- **Scope:** ink + each aux channel individually -> 5 seeds x 11 channels = **55 trainings**.
- **Config:** reuses the exact `train_uhwr_online_camera_ready_5runs.py` setup
  (functions are imported, not re-implemented).
- **Early-stopping split:** `test_leakproof`.

**Caveats**
- 55 trainings is heavy. Use `CHANNELS_TO_RUN` / `SEEDS_TO_RUN` to run in batches.
  Runs whose `best.pt` already exists are skipped (`SKIP_IF_DONE`).
- `AUX_DROPOUT_PROB=0.25` (camera-ready default) drops the only aux channel ~25%
  of steps (~ ink-only batches) as regularization. Set to `0.0` for a pure
  ink+channel model.
- **Smoke test first:** set `CHANNELS_TO_RUN=["img_dx"]`, `SEEDS_TO_RUN=[42]`,
  `MAX_EPOCHS=1` and confirm one epoch trains and
  `runs_online_single_channel_camera_ready/img_dx/seed_42/{epoch_0.pt,best.pt,artifacts/...}` appear.


In [ ]:
# --- chdir BEFORE imports so `model.*` and the trainer module resolve ---
import os
ROOT = "C:/AliCode/ICPR/main_repo"
os.chdir(ROOT)

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp.grad_scaler import GradScaler
import pandas as pd

from model.joint_multi_modality_model import JointMultiModel
from utils.dataset import ocollate_fn
from utils.dataset import RCOHWRDataset as OHWRDataset
from utils.losses import JointLoss
from tokeniser import get_tokenizer

# Reuse the camera-ready trainer's functions (importing only runs top-level
# imports + `cer = load("cer")`; main() is __main__-guarded).
from train_uhwr_online_camera_ready_5runs import (
    train_one_epoch, evaluate, save_loss_plot, set_seed, seed_worker,
)

In [ ]:
# --- Config ---
DATA_ROOT = "C:/AliCode/Datasets/data_online_line_width_alpha"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEEDS = [42, 123, 456, 789, 2024]
DATALOADER_SEED = 42  # fixed dataloader ordering across every run

img_feat = "img_stroke"
AUX_CHANNELS = [
    "img_dx",
    "img_dy",
    "img_sin_theta",
    "img_cos_theta",
    "img_curvature",
    "img_speed",
    "img_acceleration",
    "img_time_norm",
    "img_pressure",
    "img_x_tilt",
    "img_y_tilt",
]

OUT_ROOT = "runs_online_single_channel_camera_ready"

# --- Sweep knobs: subset / resume the 55-run sweep ---
CHANNELS_TO_RUN = AUX_CHANNELS      # e.g. ["img_dx"] for a smoke test
SEEDS_TO_RUN = SEEDS                # e.g. [42] for a smoke test
SKIP_IF_DONE = True                 # skip a run if its best.pt already exists

# --- Training hyperparams (reuse camera-ready config) ---
MAX_EPOCHS = 300
PATIENCE = 10
BATCH_SIZE = 8
LR = 3e-4

# Aux dropout (camera-ready default). With a single aux channel, 0.25 means
# ~25% of steps drop the only aux channel (~ ink-only). Set 0.0 for a pure model.
AUX_DROPOUT_PROB = 0.25
AUX_GROUP_DROPOUT_PROB = 0.15
MAIN_DROPOUT_PROB = 0.15

n_runs = len(CHANNELS_TO_RUN) * len(SEEDS_TO_RUN)
print(f"device={device} | {len(CHANNELS_TO_RUN)} channels x {len(SEEDS_TO_RUN)} seeds = {n_runs} runs")

In [ ]:
# --- Data (shared across all runs) ---
train_df = pd.read_csv(f"{DATA_ROOT}/train_leakproof.csv")
eval_df = pd.read_csv(f"{DATA_ROOT}/test_leakproof.csv")
tokenizer = get_tokenizer()
print(f"train={len(train_df)} | eval(test_leakproof)={len(eval_df)}")

In [ ]:
def train_one_run(channel, seed):
    """Train a fresh ink+`channel` model for one (channel, seed). Mirrors
    run_experiment in train_uhwr_online_camera_ready_5runs but with aux_feat=[channel]."""
    set_seed(seed)
    # Re-seed a fresh generator per run so dataloader ordering is identical everywhere.
    g = torch.Generator()
    g.manual_seed(DATALOADER_SEED)

    aux_feat = [channel]

    run_dir = os.path.join(OUT_ROOT, channel, f"seed_{seed}")
    artifacts_dir = os.path.join(run_dir, "artifacts")
    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(artifacts_dir, exist_ok=True)

    best_path = os.path.join(run_dir, "best.pt")
    if SKIP_IF_DONE and os.path.exists(best_path):
        print(f"[skip] {channel} seed={seed} (best.pt exists)")
        return None, None

    # Reuse existing per-feature caches (train_cache3 / test_cache3) — the single
    # channel is already cached from the full runs, so nothing is recomputed.
    train_dataset = OHWRDataset(
        DATA_ROOT, train_df, tokenizer, aug=True,
        img_feat=img_feat, aux_feat=aux_feat,
        cache_dir="train_cache3", training=True,
        aux_dropout_prob=AUX_DROPOUT_PROB,
        aux_group_dropout_prob=AUX_GROUP_DROPOUT_PROB,
        main_dropout_prob=MAIN_DROPOUT_PROB,
    )
    train_dataset.build_cache()
    val_dataset = OHWRDataset(
        DATA_ROOT, eval_df, tokenizer, aug=False,
        img_feat=img_feat, aux_feat=aux_feat,
        cache_dir="test_cache3", training=False,
    )
    val_dataset.build_cache()

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=ocollate_fn, num_workers=4,
        worker_init_fn=seed_worker, generator=g,
    )
    eval_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=ocollate_fn, num_workers=4,
        worker_init_fn=seed_worker, generator=g,
    )

    model = JointMultiModel(
        trans_enc_d_model=256,
        trans_enc_nhead=8,
        trans_enc_layers=3,
        trans_enc_ff_dim=1024,
        tokenizer=tokenizer,
        trans_dec_d_model=256,
        trans_dec_nhead=8,
        trans_dec_layers=3,
        trans_dec_n_positions=512,
        freeze_decoder=True,
        encoder_path="partials/best_transformer_encoder_uhwr_icdar.pt",
        decoder_path="partials/best_transformer_decoder_uhwr_icdar.pt",
        cnn_encoder_path="partials/best_cnn_encoder_uhwr_icdar.pt",
        ctc_head_path="partials/best_ctc_head_uhwr_icdar.pt",
        img_feat=img_feat,
        aux_feat=aux_feat,
        freeze_pcnn_encoder=False,
        freeze_tr_encoder=True,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    loss_fn = JointLoss(blank_id=tokenizer.vocab_size, ctc_weight=0.8, ce_weight=0.2)
    scaler = GradScaler()

    best_loss = float("inf")
    best_cer = float("inf")
    patience_ctr = 0
    train_loss_history = []
    val_loss_history = []

    for epoch in range(MAX_EPOCHS):
        print(f"\n[{channel} | seed {seed}] Epoch {epoch+1}/{MAX_EPOCHS}")

        train_loss = train_one_epoch(
            model, train_loader, optimizer, loss_fn, scaler, tokenizer, device
        )
        print(f"Train Loss: {train_loss:.4f}")
        train_loss_history.append(train_loss)

        val_loss, val_cer = evaluate(
            model, eval_loader, tokenizer, device, loss_fn, decode_mode="greedy"
        )
        print(f"Val Loss: {val_loss:.4f} | Val CER: {val_cer:.4f}")
        val_loss_history.append(val_loss)

        improved = False
        if val_loss < best_loss - 1e-4:
            improved = True
        elif val_cer < best_cer:
            improved = True

        torch.save(model.state_dict(), os.path.join(run_dir, f"epoch_{epoch}.pt"))
        if improved:
            best_loss = min(best_loss, val_loss)
            best_cer = min(best_cer, val_cer)
            patience_ctr = 0
            torch.save(model.state_dict(), best_path)
            print("New best model saved")
        else:
            patience_ctr += 1
            print(f"No improvement ({patience_ctr}/{PATIENCE})")

        if patience_ctr >= PATIENCE:
            print("Early stopping triggered")
            break

        plot_path = save_loss_plot(
            train_loss_history, val_loss_history, artifacts_dir,
            f"loss_curve_{channel}_seed_{seed}.png",
        )
        if plot_path:
            print(f"Loss curve saved to {plot_path}")

    print(f"[{channel} | seed {seed}] Done. Best Loss: {best_loss:.4f} | Best CER: {best_cer:.4f}")
    return best_loss, best_cer

In [ ]:
# --- Sweep: channel x seed ---
results = {}  # (channel, seed) -> best_cer
for channel in CHANNELS_TO_RUN:
    for seed in SEEDS_TO_RUN:
        print(f"\n================ {channel} | seed={seed} ================")
        best_loss, best_cer = train_one_run(channel, seed)
        results[(channel, seed)] = best_cer

        print("\n---- running summary (best CER) ----")
        for (c, s), v in results.items():
            shown = "skipped" if v is None else f"{v:.4f}"
            print(f"  {c} seed={s}: {shown}")

In [ ]:
# --- Save summary.csv (rows=channel, cols=seed + mean) ---
os.makedirs(OUT_ROOT, exist_ok=True)
summary_path = os.path.join(OUT_ROOT, "summary.csv")

rows = []
for channel in AUX_CHANNELS:
    row = {"channel": channel}
    vals = []
    for seed in SEEDS:
        v = results.get((channel, seed))
        row[f"seed_{seed}"] = v
        if v is not None:
            vals.append(v)
    row["mean"] = (sum(vals) / len(vals)) if vals else None
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(summary_path, index=False)
print(f"Saved {summary_path}")
summary_df